# BahuSanket · 20b · Pack the models for your laptop

Runs on **CPU** (no GPU needed, about 3–5 minutes). It copies the trained models, the code and the avatar files from your Drive into one
file, `BahuSanket/live_bundle/bahusanket_bundle.zip`. Download that file from Google Drive into your **Downloads** folder, then run
`bash setup.sh` in the `live_local` project (see its README).

The frozen DistilBERT part of two checkpoints is left out (setup.sh downloads the identical original), which saves about 500 MB.
**How to run:** Runtime → Run all.

In [ ]:
# 1. Find the BahuSanket folder on Drive
import os, glob, json, zipfile, shutil, base64, hashlib, urllib.request, time
import torch
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/BahuSanket"
if not os.path.isdir(ROOT):
    hits = glob.glob("/content/drive/MyDrive/**/BahuSanket", recursive=True)
    assert hits, "Could not find the BahuSanket folder in your Drive"
    ROOT = hits[0]
print("Drive folder:", ROOT)

In [ ]:
# 2. Pack
EMBEDDED = {"bahusanket_rvq.py": "IiIiUmVzaWR1YWwgcmVmaW5lbWVudCBmb3IgdGhlIHBvc2Ugdm9jYWJ1bGFyeSAoYSAicmVzaWR1YWwgVlEiLCBhcyBpbiBNb01hc2ssIGh0dHBzOi8vYXJ4aXYub3JnL2Ficy8yMzEyLjAwMDYzKS4KClRoZSBiYXNlIHZvY2FidWxhcnkgKHZxX3YyXzEyM3B0cywgNTEyIGNvZGVzLCAxIGNvZGUgcGVyIDQgZnJhbWVzKSBpcyBLRVBUIEVYQUNUTFkgQVMgSVQgSVMsIHNvIHRoZSB0ZXh0IG1vZGVsIHRoYXQKd3JpdGVzIGJhc2UgY29kZXMgKHQyY19tb3JlXzEyM3B0cyArIGd1aWRhbmNlKSBzdGF5cyB2YWxpZC4gRm9yIGV2ZXJ5IDQtZnJhbWUgc2xvdCB3ZSBhZGQgYG5fcmVzYCBleHRyYSBjb2RlczoKY29kZSAxIGRlc2NyaWJlcyB3aGF0IHRoZSBiYXNlIGNvZGUgbGVhdmVzIG91dCAodGhlICJyZXNpZHVhbCIgPSBlbmNvZGVyIHZlY3RvciBtaW51cyBpdHMgYmFzZSBjb2RlIHZlY3RvciksCmNvZGUgMiBkZXNjcmliZXMgd2hhdCBpcyBzdGlsbCBsZWZ0IGFmdGVyIGNvZGUgMSwgYW5kIHNvIG9uLiBBIG5ldyBkZWNvZGVyIHJlYWRzIGJhc2UgKyByZXNpZHVhbCBjb2RlcyB0b2dldGhlci4KCk9ubHkgdGhlIG5ldyBkZWNvZGVyIChzdGFydGVkIGZyb20gdGhlIG9sZCBvbmUpIGFuZCB0aGUgbmV3IGNvZGVib29rcyBhcmUgdHJhaW5lZDsgdGhlIGVuY29kZXIgYW5kIHRoZSBiYXNlIGNvZGVib29rIGFyZSBmcm96ZW4uCiJRdWFudGl6ZXIgZHJvcG91dCI6IGR1cmluZyB0cmFpbmluZyBhIHJhbmRvbSBudW1iZXIgb2YgcmVzaWR1YWwgbGF5ZXJzIGlzIHN3aXRjaGVkIG9mZiwgc28gdGhlIGRlY29kZXIgYWxzbyB3b3JrcyB3aXRoCmZld2VyIGxheWVycyAodGhpcyBtYWtlcyB0aGUgbGF0ZXIgdGV4dCBtb2RlbCBtb3JlIGZvcmdpdmluZzogYSB3cm9uZyBsYXN0IGxheWVyIGNvc3RzIGxpdHRsZSkuCiIiIgppbXBvcnQgY29weQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKaW1wb3J0IGJhaHVzYW5rZXRfbGliIGFzIEIKaW1wb3J0IGJhaHVzYW5rZXRfdnEgYXMgVgoKCmNsYXNzIEVNQVF1YW50aXplcihubi5Nb2R1bGUpOgogICAgIiIiT25lIGNvZGVib29rIHVwZGF0ZWQgYnkgbW92aW5nIGF2ZXJhZ2Ugd2l0aCBjb2RlIHJlc2V0IChzYW1lIHJlY2lwZSBhcyB0aGUgYmFzZSB2b2NhYnVsYXJ5KS4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBuX2NvZGVzLCBkaW0sIG11PTAuOTkpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubl9jb2Rlcywgc2VsZi5kaW0sIHNlbGYubXUgPSBuX2NvZGVzLCBkaW0sIG11CiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImNvZGVib29rIiwgdG9yY2guemVyb3Mobl9jb2RlcywgZGltKSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiY29kZV9zdW0iLCB0b3JjaC56ZXJvcyhuX2NvZGVzLCBkaW0pKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJjb2RlX2NvdW50IiwgdG9yY2gub25lcyhuX2NvZGVzKSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiaW5pdHRlZCIsIHRvcmNoLnRlbnNvcihGYWxzZSkpCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9yYW5kb21fcm93cyh6LCBuKToKICAgICAgICBpZiBsZW4oeikgPCBuOgogICAgICAgICAgICB6ID0gei5yZXBlYXQoKG4gKyBsZW4oeikgLSAxKSAvLyBsZW4oeiksIDEpCiAgICAgICAgICAgIHogPSB6ICsgdG9yY2gucmFuZG5fbGlrZSh6KSAqIDAuMDEgLyBucC5zcXJ0KHouc2hhcGVbMV0pCiAgICAgICAgcmV0dXJuIHpbdG9yY2gucmFuZHBlcm0obGVuKHopLCBkZXZpY2U9ei5kZXZpY2UpWzpuXV0KCiAgICBkZWYgZm9yd2FyZChzZWxmLCB6KToKICAgICAgICAiIiJ6IChOLCBkaW0pIGZsb2F0MzIgLT4gKGlkeCAoTiwpLCBxdWFudGlzZWQgKE4sIGRpbSksIHN0YXRzKSIiIgogICAgICAgIGlmIHNlbGYudHJhaW5pbmcgYW5kIG5vdCBib29sKHNlbGYuaW5pdHRlZCk6CiAgICAgICAgICAgIGluaXQgPSBzZWxmLl9yYW5kb21fcm93cyh6LmRldGFjaCgpLCBzZWxmLm5fY29kZXMpCiAgICAgICAgICAgIHNlbGYuY29kZWJvb2suY29weV8oaW5pdCk7IHNlbGYuY29kZV9zdW0uY29weV8oaW5pdCk7IHNlbGYuY29kZV9jb3VudC5maWxsXygxLjApCiAgICAgICAgICAgIHNlbGYuaW5pdHRlZC5maWxsXyhUcnVlKQogICAgICAgIGQgPSB6LnBvdygyKS5zdW0oMSwga2VlcGRpbT1UcnVlKSAtIDIgKiB6IEAgc2VsZi5jb2RlYm9vay50KCkgKyBzZWxmLmNvZGVib29rLnBvdygyKS5zdW0oMSlbTm9uZV0KICAgICAgICBpZHggPSBkLmFyZ21pbigxKQogICAgICAgIGlmIHNlbGYudHJhaW5pbmc6CiAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICAgICAgb25laG90ID0gRi5vbmVfaG90KGlkeCwgc2VsZi5uX2NvZGVzKS50eXBlKHouZHR5cGUpCiAgICAgICAgICAgICAgICBjbnQsIHMgPSBvbmVob3Quc3VtKDApLCBvbmVob3QudCgpIEAgei5kZXRhY2goKQogICAgICAgICAgICAgICAgc2VsZi5jb2RlX3N1bS5tdWxfKHNlbGYubXUpLmFkZF8ocywgYWxwaGE9MSAtIHNlbGYubXUpCiAgICAgICAgICAgICAgICBzZWxmLmNvZGVfY291bnQubXVsXyhzZWxmLm11KS5hZGRfKGNudCwgYWxwaGE9MSAtIHNlbGYubXUpCiAgICAgICAgICAgICAgICB1c2VkID0gKHNlbGYuY29kZV9jb3VudCA+PSAxLjApLmZsb2F0KClbOiwgTm9uZV0KICAgICAgICAgICAgICAgIGZyZXNoID0gc2VsZi5fcmFuZG9tX3Jvd3Moei5kZXRhY2goKSwgc2VsZi5uX2NvZGVzKQogICAgICAgICAgICAgICAgc2VsZi5jb2RlYm9vay5jb3B5Xyh1c2VkICogc2VsZi5jb2RlX3N1bSAvIHNlbGYuY29kZV9jb3VudFs6LCBOb25lXS5jbGFtcChtaW49MWUtNSkgKyAoMSAtIHVzZWQpICogZnJlc2gpCiAgICAgICAgICAgICAgICBzZWxmLmNvZGVfc3VtLmNvcHlfKHVzZWQgKiBzZWxmLmNvZGVfc3VtICsgKDEgLSB1c2VkKSAqIGZyZXNoKQogICAgICAgICAgICAgICAgc2VsZi5jb2RlX2NvdW50LmNvcHlfKHVzZWRbOiwgMF0gKiBzZWxmLmNvZGVfY291bnQgKyAoMSAtIHVzZWRbOiwgMF0pKQogICAgICAgIHByb2IgPSB0b3JjaC5iaW5jb3VudChpZHgsIG1pbmxlbmd0aD1zZWxmLm5fY29kZXMpLmZsb2F0KCkgLyBsZW4oaWR4KQogICAgICAgIHBlcnAgPSB0b3JjaC5leHAoLShwcm9iICogdG9yY2gubG9nKHByb2IgKyAxZS0xMCkpLnN1bSgpKQogICAgICAgIHJldHVybiBpZHgsIHNlbGYuY29kZWJvb2tbaWR4XSwgeyJwZXJwbGV4aXR5IjogcGVycCwgInVzZWQiOiAocHJvYiA+IDApLmZsb2F0KCkubWVhbigpfQoKCmNsYXNzIFJlc2lkdWFsUG9zZVZRKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgYmFzZSwgbl9yZXM9Mywgbl9jb2Rlcz01MTIpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYmFzZSA9IGJhc2UuZXZhbCgpCiAgICAgICAgZm9yIHAgaW4gc2VsZi5iYXNlLnBhcmFtZXRlcnMoKToKICAgICAgICAgICAgcC5yZXF1aXJlc19ncmFkXyhGYWxzZSkKICAgICAgICBzZWxmLmRlY29kZXIgPSBjb3B5LmRlZXBjb3B5KGJhc2UuZGVjb2RlcikgICAgICAgICAgICAjIHN0YXJ0cyBhcyB0aGUgb2xkIGRlY29kZXI7IHRyYWluZWQKICAgICAgICBmb3IgcCBpbiBzZWxmLmRlY29kZXIucGFyYW1ldGVycygpOgogICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKFRydWUpCiAgICAgICAgc2VsZi5uX3JlcyA9IG5fcmVzCiAgICAgICAgc2VsZi5yZXMgPSBubi5Nb2R1bGVMaXN0KFtFTUFRdWFudGl6ZXIobl9jb2RlcywgYmFzZS5jb2RlX2RpbSkgZm9yIF8gaW4gcmFuZ2Uobl9yZXMpXSkKCiAgICBkZWYgdHJhaW4oc2VsZiwgbW9kZT1UcnVlKToKICAgICAgICBzdXBlcigpLnRyYWluKG1vZGUpCiAgICAgICAgc2VsZi5iYXNlLmV2YWwoKSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyB0aGUgYmFzZSB2b2NhYnVsYXJ5IG5ldmVyIGNoYW5nZXMKICAgICAgICByZXR1cm4gc2VsZgoKICAgIGRlZiB0cmFpbmFibGVfcGFyYW1ldGVycyhzZWxmKToKICAgICAgICByZXR1cm4gbGlzdChzZWxmLmRlY29kZXIucGFyYW1ldGVycygpKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBfbGF0ZW50cyhzZWxmLCB4KToKICAgICAgICB6ID0gc2VsZi5iYXNlLmVuY29kZXIoeC50cmFuc3Bvc2UoMSwgMikpICAgICAgICAgICAgICAjIChCLCBDLCBUcSkKICAgICAgICBCbiwgQywgVHEgPSB6LnNoYXBlCiAgICAgICAgemYgPSB6LnRyYW5zcG9zZSgxLCAyKS5yZXNoYXBlKC0xLCBDKS5mbG9hdCgpCiAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdChkZXZpY2VfdHlwZT16Zi5kZXZpY2UudHlwZSwgZW5hYmxlZD1GYWxzZSk6CiAgICAgICAgICAgIGlkeDAsIHpxMCwgXyA9IHNlbGYuYmFzZS5xdWFudGl6ZSh6ZikgICAgICAgICAgICAgIyBiYXNlIGlzIGluIGV2YWwgbW9kZTogbm8gY29kZWJvb2sgY2hhbmdlCiAgICAgICAgcmV0dXJuIHpmLCBpZHgwLCB6cTAsIEJuLCBUcQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgsIGtlZXA9Tm9uZSk6CiAgICAgICAgIiIieCAoQiwgVCwgSU5fRElNKS4ga2VlcDogKEIsKSBudW1iZXIgb2YgcmVzaWR1YWwgbGF5ZXJzIGtlcHQgcGVyIGNsaXAgKE5vbmUgPSBhbGwpLiBSZXR1cm5zIHJlY29uLCBzdGF0cywgY29kZXMuIiIiCiAgICAgICAgemYsIGlkeDAsIHpxMCwgQm4sIFRxID0gc2VsZi5fbGF0ZW50cyh4KQogICAgICAgIHIsIHRvdGFsLCBzdGF0cywgY29kZXMgPSB6ZiAtIHpxMCwgenEwLmNsb25lKCksIHt9LCBbaWR4MF0KICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KGRldmljZV90eXBlPXpmLmRldmljZS50eXBlLCBlbmFibGVkPUZhbHNlKToKICAgICAgICAgICAgZm9yIGwsIHF6IGluIGVudW1lcmF0ZShzZWxmLnJlcyk6CiAgICAgICAgICAgICAgICBpZHgsIHEsIHN0ID0gcXoocikKICAgICAgICAgICAgICAgIHIgPSByIC0gcQogICAgICAgICAgICAgICAgZ2F0ZSA9IHRvcmNoLm9uZXMobGVuKHpmKSwgMSwgZGV2aWNlPXpmLmRldmljZSkgaWYga2VlcCBpcyBOb25lIGVsc2UgXAogICAgICAgICAgICAgICAgICAgIChsIDwga2VlcCkuZmxvYXQoKS5yZXBlYXRfaW50ZXJsZWF2ZShUcSlbOiwgTm9uZV0KICAgICAgICAgICAgICAgIHRvdGFsID0gdG90YWwgKyBxICogZ2F0ZQogICAgICAgICAgICAgICAgY29kZXMuYXBwZW5kKGlkeCkKICAgICAgICAgICAgICAgIHN0YXRzW2YicGVycHtsICsgMX0iXSwgc3RhdHNbZiJ1c2Vke2wgKyAxfSJdID0gc3RbInBlcnBsZXhpdHkiXSwgc3RbInVzZWQiXQogICAgICAgICAgICAgICAgc3RhdHNbZiJyZXN7bCArIDF9Il0gPSByLnBvdygyKS5tZWFuKCkgICAgICAgICMgaG93IG11Y2ggaXMgc3RpbGwgdW5leHBsYWluZWQgYWZ0ZXIgbGF5ZXIgbCsxCiAgICAgICAgc3RhdHNbInJlczAiXSA9ICh6ZiAtIHpxMCkucG93KDIpLm1lYW4oKQogICAgICAgIG91dCA9IHNlbGYuZGVjb2Rlcih0b3RhbC52aWV3KEJuLCBUcSwgLTEpLnRyYW5zcG9zZSgxLCAyKSkudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgcmV0dXJuIG91dCwgc3RhdHMsIHRvcmNoLnN0YWNrKFtjLnZpZXcoQm4sIFRxKSBmb3IgYyBpbiBjb2Rlc10sIC0xKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBlbmNvZGUoc2VsZiwgeCk6CiAgICAgICAgIiIiLT4gY29kZXMgKEIsIFQvNCwgMSArIG5fcmVzKTsgY29sdW1uIDAgaXMgdGhlIHVuY2hhbmdlZCBiYXNlIGNvZGUuIiIiCiAgICAgICAgd2FzID0gc2VsZi50cmFpbmluZwogICAgICAgIHNlbGYuZXZhbCgpCiAgICAgICAgXywgXywgY29kZXMgPSBzZWxmLmZvcndhcmQoeCkKICAgICAgICBzZWxmLnRyYWluKHdhcykKICAgICAgICByZXR1cm4gY29kZXMKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgZGVjb2RlKHNlbGYsIGNvZGVzLCBuX3Jlcz1Ob25lKToKICAgICAgICAiIiJjb2RlcyAoQiwgVHEsIDEgKyBuX3JlcykgLT4gcG9zZXMgKEIsIFQsIElOX0RJTSksIHVzaW5nIHRoZSBiYXNlIGNvZGUgcGx1cyB0aGUgZmlyc3Qgbl9yZXMgcmVzaWR1YWwgY29kZXMuIiIiCiAgICAgICAgbl9yZXMgPSBzZWxmLm5fcmVzIGlmIG5fcmVzIGlzIE5vbmUgZWxzZSBuX3JlcwogICAgICAgIHpxID0gc2VsZi5iYXNlLmNvZGVib29rW2NvZGVzWy4uLiwgMF1dCiAgICAgICAgZm9yIGwgaW4gcmFuZ2Uobl9yZXMpOgogICAgICAgICAgICB6cSA9IHpxICsgc2VsZi5yZXNbbF0uY29kZWJvb2tbY29kZXNbLi4uLCBsICsgMV1dCiAgICAgICAgcmV0dXJuIHNlbGYuZGVjb2Rlcih6cS50cmFuc3Bvc2UoMSwgMikpLnRyYW5zcG9zZSgxLCAyKQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIHRva2VuaXplX2FuZF9yZWJ1aWxkX3J2cShtb2RlbCwgc3BsaXQsIGlkeCwgZGV2aWNlLCBiYXRjaF9zaXplPTY0LCBuX3Jlc19saXN0PU5vbmUpOgogICAgIiIiV2hvbGUgY2xpcHMgLT4gY29kZXMgKGxpc3Qgb2YgKG5xLCAxK25fcmVzKSBpbnQxNikgYW5kIHJlYnVpbHQgcG9zZXMgZm9yIGVhY2ggbl9yZXMgaW4gbl9yZXNfbGlzdC4iIiIKICAgIG1vZGVsLmV2YWwoKQogICAgbl9yZXNfbGlzdCA9IFttb2RlbC5uX3Jlc10gaWYgbl9yZXNfbGlzdCBpcyBOb25lIGVsc2Ugbl9yZXNfbGlzdAogICAgY29kZXNfb3V0LCByZWNfb3V0ID0gW10sIHtuOiBbXSBmb3IgbiBpbiBuX3Jlc19saXN0fQogICAgZm9yIHMgaW4gcmFuZ2UoMCwgbGVuKGlkeCksIGJhdGNoX3NpemUpOgogICAgICAgIGNodW5rID0gaWR4W3M6cyArIGJhdGNoX3NpemVdCiAgICAgICAgTHMgPSBbbGVuKHNwbGl0WyJwb3NlcyJdW2ldKSBmb3IgaSBpbiBjaHVua10KICAgICAgICBUID0gaW50KG5wLmNlaWwobWF4KExzKSAvIFYuRE9XTikgKiBWLkRPV04pCiAgICAgICAgeCA9IG5wLnplcm9zKChsZW4oY2h1bmspLCBULCBWLklOX0RJTSksIG5wLmZsb2F0MzIpCiAgICAgICAgZm9yIGssIGkgaW4gZW51bWVyYXRlKGNodW5rKToKICAgICAgICAgICAgcCA9IHNwbGl0WyJwb3NlcyJdW2ldWy4uLiwgOjJdLmFzdHlwZShucC5mbG9hdDMyKS5jb3B5KCkKICAgICAgICAgICAgcFt+c3BsaXRbIm1hc2tzIl1baV1dID0gMC4wCiAgICAgICAgICAgIHhbaywgOkxzW2tdXSA9IHAucmVzaGFwZShMc1trXSwgLTEpCiAgICAgICAgY29kZXMgPSBtb2RlbC5lbmNvZGUodG9yY2guZnJvbV9udW1weSh4KS50byhkZXZpY2UpKQogICAgICAgIHJlY3MgPSB7bjogbW9kZWwuZGVjb2RlKGNvZGVzLCBuKS5mbG9hdCgpLmNwdSgpLm51bXB5KCkucmVzaGFwZShsZW4oY2h1bmspLCBULCBCLk5fUE9JTlRTLCAyKSBmb3IgbiBpbiBuX3Jlc19saXN0fQogICAgICAgIGNvZGVzID0gY29kZXMuY3B1KCkubnVtcHkoKQogICAgICAgIGZvciBrLCBMIGluIGVudW1lcmF0ZShMcyk6CiAgICAgICAgICAgIG5xID0gaW50KG5wLmNlaWwoTCAvIFYuRE9XTikpCiAgICAgICAgICAgIGNvZGVzX291dC5hcHBlbmQoY29kZXNbaywgOm5xXS5hc3R5cGUobnAuaW50MTYpKQogICAgICAgICAgICBmb3IgbiBpbiBuX3Jlc19saXN0OgogICAgICAgICAgICAgICAgcmVjX291dFtuXS5hcHBlbmQocmVjc1tuXVtrLCA6TF0pCiAgICByZXR1cm4gY29kZXNfb3V0LCByZWNfb3V0CgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgY29kZXNfdG9fcG9zZXNfcnZxKG1vZGVsLCBjb2RlX2xpc3QsIGRldmljZSwgbl9yZXM9Tm9uZSwgYmF0Y2hfc2l6ZT02NCk6CiAgICAiIiJMaXN0IG9mIChucSwgMStuX3JlcykgY29kZSBhcnJheXMgLT4gbGlzdCBvZiAoVCwgTl9QT0lOVFMsIDIpIHBvc2VzLiIiIgogICAgbW9kZWwuZXZhbCgpOyBvdXQgPSBbXQogICAgZm9yIHMgaW4gcmFuZ2UoMCwgbGVuKGNvZGVfbGlzdCksIGJhdGNoX3NpemUpOgogICAgICAgIGNoID0gY29kZV9saXN0W3M6cyArIGJhdGNoX3NpemVdCiAgICAgICAgbnEgPSBtYXgobGVuKGMpIGZvciBjIGluIGNoKQogICAgICAgIGFyciA9IG5wLnplcm9zKChsZW4oY2gpLCBucSwgY2hbMF0uc2hhcGVbMV0pLCBucC5pbnQ2NCkKICAgICAgICBmb3IgaywgYyBpbiBlbnVtZXJhdGUoY2gpOgogICAgICAgICAgICBhcnJbaywgOmxlbihjKV0gPSBjCiAgICAgICAgcmVjID0gbW9kZWwuZGVjb2RlKHRvcmNoLmZyb21fbnVtcHkoYXJyKS50byhkZXZpY2UpLCBuX3JlcykuZmxvYXQoKS5jcHUoKS5udW1weSgpLnJlc2hhcGUobGVuKGNoKSwgbnEgKiBWLkRPV04sIEIuTl9QT0lOVFMsIDIpCiAgICAgICAgb3V0ICs9IFtyZWNbaywgOmxlbihjKSAqIFYuRE9XTl0gZm9yIGssIGMgaW4gZW51bWVyYXRlKGNoKV0KICAgIHJldHVybiBvdXQK", "bahusanket_rtrans.py": "IiIiUmVzaWR1YWwtY29kZSBwcmVkaWN0b3I6IHNlbnRlbmNlICsgYmFzZSBjb2RlcyAtPiB0aGUgMyBleHRyYSBjb2RlcyBwZXIgc2xvdCAoTW9NYXNrJ3MgInJlc2lkdWFsIHRyYW5zZm9ybWVyIiwKaHR0cHM6Ly9hcnhpdi5vcmcvYWJzLzIzMTIuMDAwNjMsIFNlYy4gMy4zLCBzaW1wbGlmaWVkKS4KCkZvciBleHRyYSBsYXllciBqICgxLi4zKSB0aGUgbW9kZWwgc2VlcywgYXQgZXZlcnkgc2xvdCwgdGhlIFNVTSBvZiB0aGUgZW1iZWRkaW5ncyBvZiB0aGUgY29kZXMgb2YgbGF5ZXJzIDAuLmotMSBwbHVzIGFuCmVtYmVkZGluZyBzYXlpbmcgd2hpY2ggbGF5ZXIgaXMgd2FudGVkLCBhbmQgcHJlZGljdHMgbGF5ZXIgaidzIGNvZGUgYXQgZXZlcnkgc2xvdCBhdCBvbmNlIChubyBsZWZ0LXRvLXJpZ2h0IG9yZGVyOgp0aGUgd2hvbGUgYmFzZSBzZXF1ZW5jZSBpcyBhbHJlYWR5IGtub3duKS4gSXQgcmVhZHMgdGhlIHNlbnRlbmNlIHRoZSBzYW1lIHdheSBhcyB0aGUgdGV4dC10by1jb2RlcyBtb2RlbAooZnJvemVuIERpc3RpbEJFUlQgd29yZCB2ZWN0b3JzICsgY3Jvc3MtYXR0ZW50aW9uKS4gT25lIG1vZGVsIHNlcnZlcyBhbGwgMyBsYXllcnMuCkF0IHRlc3QgdGltZTogbGF5ZXIgMSBmcm9tIHRoZSBiYXNlIGNvZGVzLCB0aGVuIGxheWVyIDIgZnJvbSBiYXNlICsgbGF5ZXIgMSwgdGhlbiBsYXllciAzLgoiIiIKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKaW1wb3J0IGJhaHVzYW5rZXRfbGliIGFzIEIKCgpjbGFzcyBSZXNpZHVhbENvZGVQcmVkaWN0b3Iobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBuX2xheWVycz00LCBuX2NvZGVzPTUxMiwgZF9tb2RlbD0zODQsIG5faGVhZHM9Niwgbl9ibG9ja3M9NCwgZmY9MTUzNiwgZHJvcG91dD0wLjEsCiAgICAgICAgICAgICAgICAgbWF4X2xlbj0xMDAsIHRleHRfZW5jb2Rlcj0iZGlzdGlsYmVydCIsIHZvY2FiX3NpemU9MzA1MjIpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubl9sYXllcnMsIHNlbGYubl9jb2Rlcywgc2VsZi5tYXhfbGVuID0gbl9sYXllcnMsIG5fY29kZXMsIG1heF9sZW4KICAgICAgICBzZWxmLlBBRCA9IG5fY29kZXMKICAgICAgICBzZWxmLnRleHRfZW5jb2RlciA9IHRleHRfZW5jb2RlcgogICAgICAgIGlmIHRleHRfZW5jb2RlciA9PSAiZGlzdGlsYmVydCI6CiAgICAgICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwKICAgICAgICAgICAgc2VsZi5iZXJ0ID0gQXV0b01vZGVsLmZyb21fcHJldHJhaW5lZCgiZGlzdGlsYmVydC1iYXNlLXVuY2FzZWQiKQogICAgICAgICAgICBzZWxmLmJlcnQucmVxdWlyZXNfZ3JhZF8oRmFsc2UpOyBzZWxmLmJlcnQuZXZhbCgpCiAgICAgICAgICAgIHNlbGYudHh0X2luID0gbm4uTGluZWFyKHNlbGYuYmVydC5jb25maWcuZGltLCBkX21vZGVsKQogICAgICAgIGVsc2U6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyAoc21hbGwgbG9jYWwgdGVzdHMgb25seSkKICAgICAgICAgICAgc2VsZi50b2sgPSBubi5FbWJlZGRpbmcodm9jYWJfc2l6ZSwgZF9tb2RlbCwgcGFkZGluZ19pZHg9MCkKICAgICAgICBzZWxmLnR4dF9wb3MgPSBubi5FbWJlZGRpbmcoQi5NQVhfVE9LRU5TLCBkX21vZGVsKQogICAgICAgIHNlbGYuY29kZV9lbWIgPSBubi5Nb2R1bGVMaXN0KFtubi5FbWJlZGRpbmcobl9jb2RlcyArIDEsIGRfbW9kZWwsIHBhZGRpbmdfaWR4PW5fY29kZXMpIGZvciBfIGluIHJhbmdlKG5fbGF5ZXJzIC0gMSldKQogICAgICAgIHNlbGYubGF5ZXJfZW1iID0gbm4uRW1iZWRkaW5nKG5fbGF5ZXJzLCBkX21vZGVsKQogICAgICAgIHNlbGYucG9zID0gbm4uRW1iZWRkaW5nKG1heF9sZW4sIGRfbW9kZWwpCiAgICAgICAgc2VsZi5ibG9ja3MgPSBubi5UcmFuc2Zvcm1lckRlY29kZXIobm4uVHJhbnNmb3JtZXJEZWNvZGVyTGF5ZXIoCiAgICAgICAgICAgIGRfbW9kZWwsIG5faGVhZHMsIGZmLCBkcm9wb3V0LCBiYXRjaF9maXJzdD1UcnVlLCBub3JtX2ZpcnN0PVRydWUpLCBuX2Jsb2NrcykKICAgICAgICBzZWxmLm5vcm0gPSBubi5MYXllck5vcm0oZF9tb2RlbCkKICAgICAgICBzZWxmLmhlYWRzID0gbm4uTW9kdWxlTGlzdChbbm4uTGluZWFyKGRfbW9kZWwsIG5fY29kZXMpIGZvciBfIGluIHJhbmdlKG5fbGF5ZXJzIC0gMSldKSAgICMgb25lIHBlciBleHRyYSBsYXllcgogICAgICAgIHNlbGYuZHJvcCA9IG5uLkRyb3BvdXQoZHJvcG91dCkKCiAgICBkZWYgdHJhaW4oc2VsZiwgbW9kZT1UcnVlKToKICAgICAgICBzdXBlcigpLnRyYWluKG1vZGUpCiAgICAgICAgaWYgc2VsZi50ZXh0X2VuY29kZXIgPT0gImRpc3RpbGJlcnQiOgogICAgICAgICAgICBzZWxmLmJlcnQuZXZhbCgpCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgZW5jb2RlX3RleHQoc2VsZiwgaWRzLCBhdHRuKToKICAgICAgICBpZiBzZWxmLnRleHRfZW5jb2RlciA9PSAiZGlzdGlsYmVydCI6CiAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICAgICAgaCA9IHNlbGYuYmVydChpbnB1dF9pZHM9aWRzLCBhdHRlbnRpb25fbWFzaz1hdHRuKS5sYXN0X2hpZGRlbl9zdGF0ZQogICAgICAgICAgICBoID0gc2VsZi50eHRfaW4oaC5mbG9hdCgpKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGggPSBzZWxmLnRvayhpZHMpCiAgICAgICAgcmV0dXJuIHNlbGYuZHJvcChoICsgc2VsZi50eHRfcG9zKHRvcmNoLmFyYW5nZShpZHMuc2hhcGVbMV0sIGRldmljZT1pZHMuZGV2aWNlKSkpLCBhdHRuID09IDAKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBjb2RlcywgaiwgbWVtLCBtZW1fcGFkLCBwYWQpOgogICAgICAgICIiImNvZGVzIChCLCBULCBuX2xheWVycykgd2l0aCBQQUQgd2hlcmUgbWlzc2luZzsgaiAoQiwpIHRhcmdldCBsYXllciBpbiAxLi5uX2xheWVycy0xIC0+IGxvZ2l0cyAoQiwgVCwgbl9jb2RlcykuIiIiCiAgICAgICAgQm4sIFQsIF8gPSBjb2Rlcy5zaGFwZQogICAgICAgIGggPSBzZWxmLnBvcyh0b3JjaC5hcmFuZ2UoVCwgZGV2aWNlPWNvZGVzLmRldmljZSkpW05vbmVdICsgc2VsZi5sYXllcl9lbWIoailbOiwgTm9uZV0KICAgICAgICBmb3IgbCBpbiByYW5nZShzZWxmLm5fbGF5ZXJzIC0gMSk6ICAgICAgICAgICMgc3VtIG9mIHRoZSBlbWJlZGRpbmdzIG9mIHRoZSBsYXllcnMgYmVsb3cgagogICAgICAgICAgICB1c2UgPSAobCA8IGopLmZsb2F0KClbOiwgTm9uZSwgTm9uZV0KICAgICAgICAgICAgaCA9IGggKyBzZWxmLmNvZGVfZW1iW2xdKGNvZGVzWy4uLiwgbF0pICogdXNlCiAgICAgICAgaCA9IHNlbGYuYmxvY2tzKHNlbGYuZHJvcChoKSwgbWVtLCB0Z3Rfa2V5X3BhZGRpbmdfbWFzaz1wYWQsIG1lbW9yeV9rZXlfcGFkZGluZ19tYXNrPW1lbV9wYWQpCiAgICAgICAgaCA9IHNlbGYubm9ybShoKQogICAgICAgIG91dCA9IHRvcmNoLnN0YWNrKFtoZChoKSBmb3IgaGQgaW4gc2VsZi5oZWFkc10sIDEpICAgICAgICAgICAgICAgICAgICAgICAjIChCLCBuX2xheWVycy0xLCBULCBuX2NvZGVzKQogICAgICAgIHJldHVybiBvdXRbdG9yY2guYXJhbmdlKEJuLCBkZXZpY2U9Y29kZXMuZGV2aWNlKSwgaiAtIDFdCgogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIGZpbGwoc2VsZiwgaWRzLCBhdHRuLCBiYXNlLCB0ZW1wZXJhdHVyZT0wLjApOgogICAgICAgICIiImJhc2U6IGxpc3Qgb2YgMS1EIGJhc2UtY29kZSBhcnJheXMgLT4gbGlzdCBvZiAobGVuLCBuX2xheWVycykgYXJyYXlzLiB0ZW1wZXJhdHVyZSAwID0gbW9zdCBsaWtlbHkgY29kZS4iIiIKICAgICAgICBuLCBUID0gbGVuKGJhc2UpLCBtaW4oc2VsZi5tYXhfbGVuLCBtYXgobGVuKGIpIGZvciBiIGluIGJhc2UpKQogICAgICAgIGRldiA9IGlkcy5kZXZpY2UKICAgICAgICBjb2RlcyA9IHRvcmNoLmZ1bGwoKG4sIFQsIHNlbGYubl9sYXllcnMpLCBzZWxmLlBBRCwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPWRldikKICAgICAgICBmb3IgaywgYiBpbiBlbnVtZXJhdGUoYmFzZSk6CiAgICAgICAgICAgIGNvZGVzW2ssIDptaW4obGVuKGIpLCBUKSwgMF0gPSB0b3JjaC5hc190ZW5zb3IobnAuYXNhcnJheShiWzpUXSwgbnAuaW50NjQpLCBkZXZpY2U9ZGV2KQogICAgICAgIHBhZCA9IGNvZGVzWy4uLiwgMF0gPT0gc2VsZi5QQUQKICAgICAgICBtZW0sIG1lbV9wYWQgPSBzZWxmLmVuY29kZV90ZXh0KGlkcywgYXR0bikKICAgICAgICBmb3IgaiBpbiByYW5nZSgxLCBzZWxmLm5fbGF5ZXJzKToKICAgICAgICAgICAgbGcgPSBzZWxmLmZvcndhcmQoY29kZXMsIHRvcmNoLmZ1bGwoKG4sKSwgaiwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPWRldiksIG1lbSwgbWVtX3BhZCwgcGFkKS5mbG9hdCgpCiAgICAgICAgICAgIGlmIHRlbXBlcmF0dXJlID4gMDoKICAgICAgICAgICAgICAgIG54dCA9IHRvcmNoLm11bHRpbm9taWFsKEYuc29mdG1heChsZyAvIHRlbXBlcmF0dXJlLCAtMSkudmlldygtMSwgc2VsZi5uX2NvZGVzKSwgMSkudmlldyhuLCBUKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgbnh0ID0gbGcuYXJnbWF4KC0xKQogICAgICAgICAgICBjb2Rlc1suLi4sIGpdID0gdG9yY2gud2hlcmUocGFkLCB0b3JjaC5mdWxsX2xpa2Uobnh0LCBzZWxmLlBBRCksIG54dCkKICAgICAgICBjb2RlcyA9IGNvZGVzLmNwdSgpLm51bXB5KCkKICAgICAgICBvdXQgPSBbXQogICAgICAgIGZvciBrLCBiIGluIGVudW1lcmF0ZShiYXNlKToKICAgICAgICAgICAgYyA9IGNvZGVzW2ssIDptaW4obGVuKGIpLCBUKV0KICAgICAgICAgICAgaWYgbGVuKGIpID4gVDogICAgICAgICAgICAgICAgICAgICAgICAgICAjIChsb25nZXIgdGhhbiB0aGUgbW9kZWwgY2FuIHNlZTogcmVwZWF0IHRoZSBsYXN0IGV4dHJhcykKICAgICAgICAgICAgICAgIGMgPSBucC5jb25jYXRlbmF0ZShbYywgbnAuY29uY2F0ZW5hdGUoW25wLmFzYXJyYXkoYltUOl0sIG5wLmludDY0KVs6LCBOb25lXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5wLnJlcGVhdChjWy0xOiwgMTpdLCBsZW4oYikgLSBULCAwKV0sIDEpXSkKICAgICAgICAgICAgb3V0LmFwcGVuZChjLmFzdHlwZShucC5pbnQ2NCkpCiAgICAgICAgcmV0dXJuIG91dAoKCmRlZiByZXNpZHVhbF9iYXRjaChjb2Rlc19saXN0LCBuX2xheWVycywgcGFkX2lkLCBtYXhfbGVuLCBjb3JydXB0PTAuMCwgal9maXhlZD1Ob25lKToKICAgICIiIkJhdGNoIG9mIChsZW4sIG5fbGF5ZXJzKSBjb2RlIGFycmF5cyAtPiBjb2RlcyAoQiwgVCwgTCkgd2l0aCBQQUQsIHRhcmdldCBsYXllciBqIChCLCksIHRhcmdldHMgKEIsIFQpICgtMTAwID0gaWdub3JlKS4iIiIKICAgIFQgPSBtaW4obWF4X2xlbiwgbWF4KGxlbihjKSBmb3IgYyBpbiBjb2Rlc19saXN0KSkKICAgIG4gPSBsZW4oY29kZXNfbGlzdCkKICAgIHggPSBucC5mdWxsKChuLCBULCBuX2xheWVycyksIHBhZF9pZCwgbnAuaW50NjQpCiAgICBqID0gbnAucmFuZG9tLnJhbmRpbnQoMSwgbl9sYXllcnMsIG4pIGlmIGpfZml4ZWQgaXMgTm9uZSBlbHNlIG5wLmZ1bGwobiwgal9maXhlZCkKICAgIHRndCA9IG5wLmZ1bGwoKG4sIFQpLCAtMTAwLCBucC5pbnQ2NCkKICAgIGZvciBrLCBjIGluIGVudW1lcmF0ZShjb2Rlc19saXN0KToKICAgICAgICBjID0gbnAuYXNhcnJheShjLCBucC5pbnQ2NClbOlRdCiAgICAgICAgeFtrLCA6bGVuKGMpXSA9IGMKICAgICAgICB0Z3RbaywgOmxlbihjKV0gPSBjWzosIGpba11dCiAgICAgICAgaWYgY29ycnVwdCA+IDAgYW5kIGxlbihjKTogICAgICAgICAgICAgICAgICAgIyByZXBsYWNlIGEgZmV3IGlucHV0IGNvZGVzIHdpdGggcmFuZG9tIG9uZXMgKHJvYnVzdG5lc3MgdG8gZ2VuZXJhdGVkIGJhc2UgY29kZXMpCiAgICAgICAgICAgIGhpdCA9IG5wLnJhbmRvbS5yYW5kKGxlbihjKSwgbl9sYXllcnMpIDwgbnAucmFuZG9tLnVuaWZvcm0oMCwgY29ycnVwdCkKICAgICAgICAgICAgaGl0WzosIGpba106XSA9IEZhbHNlCiAgICAgICAgICAgIHhbaywgOmxlbihjKV1baGl0XSA9IG5wLnJhbmRvbS5yYW5kaW50KDAsIHBhZF9pZCwgaGl0LnN1bSgpKQogICAgdCA9IHRvcmNoLmFzX3RlbnNvcgogICAgcmV0dXJuIHQoeCksIHQoaiksIHQodGd0KQo="}
OUT = "/content/bundle"; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(f"{OUT}/checkpoints"); os.makedirs(f"{OUT}/code")
CK = os.path.join(ROOT, "checkpoints")
KEEP = {"model", "config", "mu", "sd", "zmean", "zstd", "epoch", "step"}
for src, dst, drop_bert in [("vq_v2_123pts/last.pt", "vq_v2_123pts.pt", False), ("rvq_v2_123pts/last.pt", "rvq_v2_123pts.pt", False),
                            ("rtrans_123pts_b/best.pt", "rtrans_123pts_b.pt", True), ("t2c_more_123pts/best_loss.pt", "t2c_more_123pts.pt", True),
                            ("length_predictor_v1.pt", "length_predictor_v1.pt", False), ("depth_lifter_v1/best.pt", "depth_lifter_v1.pt", False)]:
    ck = torch.load(os.path.join(CK, src), map_location="cpu", weights_only=False)
    small = {k: v for k, v in ck.items() if k in KEEP}
    if drop_bert:
        small["model"] = {k: v for k, v in ck["model"].items() if not k.startswith("bert.")}
    torch.save(small, f"{OUT}/checkpoints/{dst}")
    print(f"  {src:<32} -> {dst:<24} {os.path.getsize(f'{OUT}/checkpoints/{dst}') / 1e6:6.1f} MB")
for n in ("bahusanket_lib.py", "bahusanket_vq.py", "bahusanket_t2c.py", "bahusanket_layout.py"):
    shutil.copy(os.path.join(ROOT, "code", n), f"{OUT}/code/{n}")
for n, b in EMBEDDED.items():                                   # notebook 13/14b module code (the same as cell 2 of notebooks 17-20)
    open(f"{OUT}/code/{n}", "wb").write(base64.b64decode(b))
shutil.copy(os.path.join(ROOT, "model_point_indices_v2.json"), OUT)
shutil.copy(os.path.join(ROOT, "results", "12_bone_limits.json"), OUT)
tc = os.path.join(ROOT, "translations19", "cache.json")
if os.path.exists(tc):
    shutil.copy(tc, f"{OUT}/translations_cache.json")
AV = os.path.join(ROOT, "avatar")
for n in ("clips_paragraph_19.json", "clips_model_18.json", "clips.json"):
    if os.path.exists(os.path.join(AV, n)):
        shutil.copy(os.path.join(AV, n), OUT)
    else:
        print("  (not on Drive, skipped:", n, ")")
if os.path.exists(os.path.join(AV, "avatar.b64.txt")):
    shutil.copy(os.path.join(AV, "avatar.b64.txt"), OUT)
else:
    url = "https://raw.githubusercontent.com/pixiv/three-vrm/release/packages/three-vrm/examples/models/VRM1_Constraint_Twist_Sample.vrm"
    vrm = urllib.request.urlopen(url, timeout=120).read()
    assert hashlib.md5(vrm).hexdigest() == "b36a6fbb65316d5684de896dedee4f26", "unexpected avatar file"
    open(f"{OUT}/avatar.b64.txt", "w").write(base64.b64encode(vrm).decode())
ZIPF = os.path.join(ROOT, "live_bundle", "bahusanket_bundle.zip"); os.makedirs(os.path.dirname(ZIPF), exist_ok=True)
with zipfile.ZipFile("/content/bahusanket_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for dp, _, fs in os.walk(OUT):
        for f in fs:
            z.write(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), OUT))
shutil.copy("/content/bahusanket_bundle.zip", ZIPF)
print(f"\nsaved {ZIPF} ({os.path.getsize(ZIPF) / 1e6:.0f} MB). Download it from Google Drive into your Downloads folder.")